# MNIST: Denoising-Perceptual Distance (standalone Colab version)

Exploratory alternative to the trajectory embedding: the distance between
images is the divergence of the model's noise predictions under shared noise,

$$d^2(x_1, x_2) = \mathbb{E}_{\varepsilon, t} \|\varepsilon_\theta(\sqrt{\bar\alpha_t} x_1 + \sqrt{1-\bar\alpha_t}\varepsilon, t) - \varepsilon_\theta(\sqrt{\bar\alpha_t} x_2 + \sqrt{1-\bar\alpha_t}\varepsilon, t)\|^2,$$

plugged into the same kernel / log-det pipeline. Self-contained for a Colab
GPU runtime (no `core/` imports).

Before running, upload `ddpm_mnist_pretrained.pt`
(produced by `python export_checkpoint.py`).

In [ ]:
!pip install diffusers accelerate -q

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from diffusers import UNet2DModel
from scipy.spatial.distance import squareform

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load pretrained model
ckpt = torch.load('ddpm_mnist_pretrained.pt', map_location=device, weights_only=False)
unet = UNet2DModel(**{k: v for k, v in ckpt['unet_config'].items()
                      if k not in ('_class_name', '_diffusers_version', '_name_or_path',
                                   '_use_default_values')})
unet.load_state_dict(ckpt['unet_state_dict'])
unet = unet.to(device).eval()

# Noise schedule
T = ckpt['scheduler_config']['num_train_timesteps']
betas = np.linspace(ckpt['scheduler_config']['beta_start'],
                    ckpt['scheduler_config']['beta_end'], T)
alpha_bars = np.cumprod(1.0 - betas)
alpha_bars_t = torch.from_numpy(alpha_bars.astype(np.float32)).to(device)

print(f'Model: {sum(p.numel() for p in unet.parameters()):,} params')
print(f'Schedule: T={T}, alpha_bar range [{alpha_bars[-1]:.6f}, {alpha_bars[0]:.6f}]')

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
ds = datasets.MNIST('data', train=True, download=True, transform=transform)

by_digit = {d: [] for d in range(10)}
for img, label in ds:
    by_digit[label].append(img)
by_digit = {d: torch.stack(v) for d, v in by_digit.items()}
print('Loaded.')

## 1. Perceptual Distance at Each Noise Level

Check within-class vs between-class separation at each t.

In [ ]:
@torch.no_grad()
def eps_distance_at_t(x1, x2, k, n_mc=8):
    """
    Monte Carlo estimate of ||eps_theta(x1_t) - eps_theta(x2_t)||^2
    at discrete timestep k, averaging over shared noise realizations.
    
    x1, x2: (1, 28, 28) tensors on device
    k: discrete timestep
    n_mc: number of noise samples
    """
    abar = alpha_bars_t[k]
    sqrt_abar = torch.sqrt(abar)
    sqrt_1m_abar = torch.sqrt(1 - abar)
    
    # Batch: repeat images n_mc times
    x1_rep = x1.unsqueeze(0).expand(n_mc, -1, -1, -1)  # (n_mc, 1, 28, 28)
    x2_rep = x2.unsqueeze(0).expand(n_mc, -1, -1, -1)
    
    # Same noise for both
    eps = torch.randn_like(x1_rep)
    
    x1_t = sqrt_abar * x1_rep + sqrt_1m_abar * eps
    x2_t = sqrt_abar * x2_rep + sqrt_1m_abar * eps
    
    t_batch = torch.full((n_mc,), k, device=device, dtype=torch.long)
    
    # Batch both through model at once
    x_both = torch.cat([x1_t, x2_t], dim=0)  # (2*n_mc, 1, 28, 28)
    t_both = torch.cat([t_batch, t_batch], dim=0)
    eps_both = unet(x_both, t_both).sample
    
    eps1 = eps_both[:n_mc]
    eps2 = eps_both[n_mc:]
    
    # Mean squared difference per pixel, averaged over MC samples
    diff_sq = ((eps1 - eps2) ** 2).mean(dim=(1, 2, 3))  # (n_mc,)
    return diff_sq.mean().item()


# Test: 5 per digit = 50 images
N_per = 5
imgs = []
labs = []
for d in range(10):
    imgs.append(by_digit[d][:N_per].to(device))
    labs.extend([d] * N_per)
imgs = torch.cat(imgs)  # (50, 1, 28, 28)
labs = np.array(labs)
N = len(labs)

# Evaluate at several timesteps
k_values = [50, 100, 200, 300, 400, 500, 600, 700, 800, 900, 950]
t_cont = [k / (T - 1) for k in k_values]

within_dists = []
between_dists = []

for k in k_values:
    print(f'k={k} (t={k/(T-1):.3f}, abar={alpha_bars[k]:.4f})...')
    w_d = []
    b_d = []
    for i in range(N):
        for j in range(i + 1, N):
            d = eps_distance_at_t(imgs[i], imgs[j], k, n_mc=8)
            if labs[i] == labs[j]:
                w_d.append(d)
            else:
                b_d.append(d)
    within_dists.append(np.mean(w_d))
    between_dists.append(np.mean(b_d))
    print(f'  within={within_dists[-1]:.4f}, between={between_dists[-1]:.4f}, ratio={between_dists[-1]/within_dists[-1]:.3f}')

print('Done.')

In [ ]:
ratios = [b / w for b, w in zip(between_dists, within_dists)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(t_cont, within_dists, 'o-', color='steelblue', label='Within-class')
axes[0].plot(t_cont, between_dists, 's-', color='firebrick', label='Between-class')
axes[0].set_xlabel('t')
axes[0].set_ylabel('Mean eps-distance')
axes[0].set_title('Denoising prediction distance vs noise level')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_cont, ratios, 'o-', color='darkgreen')
axes[1].set_xlabel('t')
axes[1].set_ylabel('Between / Within ratio')
axes[1].set_title('Class separation ratio')
axes[1].axhline(1, color='gray', ls='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_idx = np.argmax(ratios)
print(f'Best separation: t={t_cont[best_idx]:.3f} (k={k_values[best_idx]}), ratio={ratios[best_idx]:.3f}')

## 2. Full Distance Matrix & Complexity

Use the best timestep (or average over several) to build the pairwise distance matrix, then compute C(X).

In [ ]:
@torch.no_grad()
def eps_distance_matrix(images, k_values, n_mc=16):
    """
    Compute pairwise eps-distance matrix averaged over timesteps and noise.
    
    images: (N, 1, 28, 28) tensor on device
    k_values: list of discrete timesteps to average over
    n_mc: noise samples per timestep
    
    Returns: (N, N) numpy distance matrix
    """
    N = images.shape[0]
    D = torch.zeros(N, N, device=device)
    
    for k in k_values:
        abar = alpha_bars_t[k]
        sqrt_abar = torch.sqrt(abar)
        sqrt_1m_abar = torch.sqrt(1 - abar)
        
        for mc in range(n_mc):
            # Same noise for all images
            eps = torch.randn(1, 1, 28, 28, device=device).expand(N, -1, -1, -1)
            x_t = sqrt_abar * images + sqrt_1m_abar * eps  # (N, 1, 28, 28)
            
            t_batch = torch.full((N,), k, device=device, dtype=torch.long)
            eps_pred = unet(x_t, t_batch).sample  # (N, 1, 28, 28)
            eps_flat = eps_pred.reshape(N, -1)  # (N, 784)
            
            # Pairwise squared distances
            # ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a.b
            norms_sq = (eps_flat ** 2).sum(dim=1)  # (N,)
            gram = eps_flat @ eps_flat.T  # (N, N)
            D_k = norms_sq.unsqueeze(1) + norms_sq.unsqueeze(0) - 2 * gram
            D += D_k.clamp(min=0)
    
    D = D / (len(k_values) * n_mc)
    D = torch.sqrt(D)  # squared -> distance
    return D.cpu().numpy()

In [ ]:
def compute_complexity(D, lambda_):
    K = np.exp(-lambda_ * D ** 2)
    N = K.shape[0]
    L = np.linalg.cholesky(np.eye(N) + K)
    return 2.0 * np.sum(np.log(np.diag(L)))

def calibrate_bandwidth(D):
    upper = D[np.triu_indices_from(D, k=1)]
    med_sq = np.median(upper ** 2)
    return 1.0 / med_sq if med_sq > 0 else 1.0

In [ ]:
# Pick timesteps to use (from separation analysis above)
# Use a few timesteps around the best separation
k_use = [100, 200, 300, 400, 500]

# Calibrate lambda from 30 held-out zeros
print('Calibrating lambda...')
ref = by_digit[0][200:230].to(device)
D_ref = eps_distance_matrix(ref, k_use, n_mc=16)
lambda_ = calibrate_bandwidth(D_ref)
print(f'lambda = {lambda_:.6f}')
print(f'Ref distances: mean={D_ref[D_ref>0].mean():.3f}, median={np.median(D_ref[D_ref>0]):.3f}')

# Growth experiment
N_values = [5, 10, 15, 20, 30, 40, 50]
C_single = []
C_multi = []

for N in N_values:
    print(f'\n--- N = {N} ---')
    
    # Single class (zeros)
    x_s = by_digit[0][:N].to(device)
    D_s = eps_distance_matrix(x_s, k_use, n_mc=16)
    c_s = compute_complexity(D_s, lambda_)
    C_single.append(c_s)
    print(f'  C_single = {c_s:.4f}')
    
    # Multi-class (cycling)
    counts = {d: 0 for d in range(10)}
    x_m = []
    for i in range(N):
        d = i % 10
        x_m.append(by_digit[d][counts[d]])
        counts[d] += 1
    x_m = torch.stack(x_m).to(device)
    
    D_m = eps_distance_matrix(x_m, k_use, n_mc=16)
    c_m = compute_complexity(D_m, lambda_)
    C_multi.append(c_m)
    print(f'  C_multi  = {c_m:.4f}')

print('\nDone.')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(N_values, C_single, 'o-', color='steelblue', lw=2, label='Single class (digit 0)')
ax.plot(N_values, C_multi, 's-', color='firebrick', lw=2, label='Multi-class (cycling 0\u20139)')
ax.set_xlabel('Sample size N', fontsize=12)
ax.set_ylabel('C(X)', fontsize=12)
ax.set_title('Dataset Complexity: Eps-Perceptual Distance', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mnist_growth_perceptual.png', dpi=150)
plt.show()

print(f'{"N":>5} {"C_single":>10} {"C_multi":>10} {"ratio":>8}')
for i, N in enumerate(N_values):
    r = C_multi[i] / C_single[i] if C_single[i] > 0 else float('inf')
    print(f'{N:>5} {C_single[i]:>10.4f} {C_multi[i]:>10.4f} {r:>8.2f}')